# Стандартизация и нормализация данных

In [9]:
# необходимые библиотеки
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, normalize, MaxAbsScaler, PowerTransformer, QuantileTransformer
from sklearn.decomposition import PCA, _pca

# Нормализация данных (MinMaxScaler)

In [ ]:
# от значения отнимает минимум и делит на максимум минус минимум - x * = (x-min) / (max-min) 
# Недостатки : Чувствительна к выбросам.

scaler_minmax = MinMaxScaler(feature_range = (0, 1))
df_scaled_minmax = scaler_minmax.fit_transform(df)

scaler_minmax = MinMaxScaler(feature_range = (-1, 1))
df_scaled_minmax = scaler_minmax.fit_transform(df)

# Описание : Преобразует данные с использованием логарифма для уменьшения влияния выбросов.
df_log = np.log1p(df)
scaler_minmax = MinMaxScaler(feature_range = (0, 1))
df_scaled_minmax = scaler_minmax.fit_transform(df_log)

# Когда использовать : Когда данные имеют известный диапазон и не содержат выбросов.
# Примеры : Нейронные сети, алгоритмы, основанные на расстоянии (например, KNN).

# Стандартизация данных (StandardScaler)

In [ ]:
# (X−μ)/σ
# Недостатки : Чувствительна к выбросам.

scaler_standard = StandardScaler(feature_range = (0, 1))
df_scaled_standard = scaler_minmax.fit_transform(df)

# Когда использовать : Когда данные имеют нормальное распределение и не содержат выбросов.
# Примеры : Линейные модели, SVM.

# RobustScaler

In [ ]:
# (X−median(X)) / IQR(X)
# Преимущества : Устойчивость к выбросам.

scaler = RobustScaler()
X_robust = scaler.fit_transform(X)

# Когда использовать : Когда данные содержат выбросы или асимметричное распределение.
# Примеры : Линейные модели, SVM.

# MaxAbsScaler

In [ ]:
# X / max(∣X∣)
# Недостатки : Чувствительна к выбросам.

scaler = MaxAbsScaler()
X_maxabs = scaler.fit_transform(X)

# Когда использовать : Когда данные не содержат нуля и не требуют центрирования.
# Примеры : Нейронные сети, алгоритмы, основанные на расстоянии (например, KNN).

# PowerTransformer (Yeo-Johnson или Box-Cox Transformation)

In [ ]:
scaler = PowerTransformer(method = 'yeo-johnson')  # или method='box-cox'
X_power_transformed = scaler.fit_transform(X)

# Когда использовать : Когда данные имеют неположительные значения или асимметричное распределение.
# Примеры : Линейные модели, SVM.

# QuantileTransformer

In [ ]:
scaler = QuantileTransformer(output_distribution='normal', random_state=42)
X_quantile_transformed = scaler.fit_transform(X)

# Когда использовать : Когда данные имеют асимметричное распределение или требуют улучшения нормальности.
# Примеры : Линейные модели, SVM, нейронные сети.

# L1 нормализация

In [ ]:
data_normalized_l1 = normalize(df, norm = 'l1')

# L2 нормализация

In [ ]:
data_normalized_l2 = normalize(df, norm = 'l2')

# Логарифмическое преобразование (Log Transformation)

In [ ]:
X['area_log'] = np.log1p(X['area']) # добавляет 1 к каждому значению, чтобы исключить 0
X['area_log'] = np.log(X['area'])

# Квадратичное преобразование, квадратный корень (Square Root Transformation)

In [ ]:
X['area_sqrt'] = np.sqrt(X['area'])

# Кубическое преобразование, кубический корень (Cube Root Transformation)

In [ ]:
X['area_cbrt'] = np.cbrt(X['area'])

# Возведение в квадрат/куб (Quadratic/Cubic Transformation)

In [ ]:
X['area_squared'] = X['area'] ** 2
X['area_cubed'] = X['area'] ** 3

# Полиномиальное преобразование (Polynomial Features)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Создаём полиномиальные признаки степени 2
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X[['area', 'distance_to_metro']])

# Преобразуем массив обратно в DataFrame с соответствующими именами колонок
feature_names = poly.get_feature_names_out(['area', 'distance_to_metro'])
X_poly_df = pd.DataFrame(X_poly, columns=feature_names)

# Объединяем новые признаки с исходными данными
X = pd.concat([X, X_poly_df], axis=1)

# Синусоидальное / Косинусоидальное преобразование (Sine/Cosine Transformation)

In [ ]:
X['area_sin'] = np.sin(X['area'])
X['area_cos'] = np.cos(X['area'])

# FeatureHasher

- признаки категориальные и имеют **много уникальных значений** (например, ID, названия улиц, юзер-агенты и т.д.),
- и при этом **не хочется использовать one-hot**, который может сильно раздувать размерность.

---

### 🔹 Пример использования `FeatureHasher`

Предположим, у нас есть такой датасет с признаками:

```python
data = [
    {'улица': 'ул. Ленина', 'материал': 'панель', 'состояние': 'требует ремонта'},
    {'улица': 'ул. Гагарина', 'материал': 'кирпич', 'состояние': 'хорошее'},
    {'улица': 'ул. Победы', 'материал': 'монолит', 'состояние': 'евроремонт'},
]
```

Теперь захешируем признаки:

```python
from sklearn.feature_extraction import FeatureHasher
import pandas as pd

# Хешер — укажем количество выходных фичей (например, 8)
hasher = FeatureHasher(n_features=8, input_type='dict')

# Преобразуем список словарей
hashed_features = hasher.transform(data)

# Получим DataFrame (если нужно посмотреть)
hashed_df = pd.DataFrame(hashed_features.toarray())

print(hashed_df)
```

---

### 🔹 Что ты получишь:

Будет 8 новых признаков с float значениями (могут быть отрицательными). Это не one-hot, а **псевдочастотное хеширование**, где один и тот же признак может попасть в одну и ту же колонку, как и другие (коллизии возможны, но их можно контролировать через `n_features`).

---

### 🔹 Применимость:

- В твоем случае, если ты хочешь **кодировать улицы, районы, жк и т.д.**, у которых **много уникальных значений**, но **не хочешь увеличивать размерность через one-hot** — `FeatureHasher` может быть отличным решением.

---

### 🔹 Подводные камни:
- Хеширование неинтерпретируемо.
- Возможны **коллизии** (разные значения попадают в один и тот же хеш).
- Лучше работает, если **уникальных значений очень много** (сотни и тысячи).

Хочешь — могу показать на реальном примере с улицами или районами из твоих данных.

## Обработка категориальных данных

In [ ]:
# необходимые библиотеки
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [ ]:
# переводит переменные в 0 и 1, а также числовые переменные в численные значения другого порядка
encoder = LabelEncoder()
df['CategoricalFare'] = encoder.fit_transform(df[['CategoricalFare']])
df.head()

In [ ]:
# переводит переменные в 0 и 1 и создает дополнительные столбцы
enc = OneHotEncoder(handle_unknown='ignore')
enc_df = pd.DataFrame(enc.fit_transform(df[['Embarked']]).toarray())

df = df.join(enc_df)
df = df.drop(['Embarked'], axis = 1)
df.reset_index()
df.head()

In [ ]:
Работа с признаками:
1. Label enc, OHE (один столбец убираем, заменяем на -1), mean enc
2. ⁠ln, степень, корень, бок кокс

4. ⁠Перемножение признаков.
5. ⁠бининг или квантование 

Для каких алгоритмов важно преобразование? Древовидные алгоритмы (деревья и бустинги) устойчивы к масштабу.

Изменение размерности:
1. Исключение признаков.
2. ⁠PCA, tsne, NCA
3. ⁠lasso
4. ⁠важность признаков

▌Общие принципы стандартизации и масштабирования

▌1. Что такое стандартизация?

- Обычно — это преобразование признаков так, чтобы они имели среднее 0 и стандартное отклонение 1 (например, через StandardScaler).
- Это важно для моделей, чувствительных к масштабу признаков, таких как нейронные сети, градиентный бустинг, SVM и др.

▌2. Типы признаков

- Бинарные признаки (0/1) — например, holiday_flag, work_saturday.
- Дискретные числовые признаки с большими значениями — например, centers, count, some_large_numbers.
- Категориальные признаки — например, week_number (если не закодировать как числа).

---

▌Как лучше обрабатывать ваши признаки?

▌1. Бинарные признаки (0/1)

- Не нужно масштабировать.
- Они уже в подходящем виде для большинства моделей, особенно для нейронных сетей и градиентных бустингов.
- Масштабировать их не имеет смысла, так как они уже в диапазоне [0,1].

▌2. Дискретные признаки с большими значениями

- Да, их лучше стандартизировать.
- Почему? 
 - Для нейронных сетей — это помогает стабилизировать обучение, ускоряет сходимость.
 - Для градиентных бустингов — обычно не обязательно, но иногда помогает.

▌3. Категориальные признаки

- Обычно закодируют через one-hot encoding или embedding (для нейронных сетей).
- После кодирования — масштабировать не нужно.

---

▌Итоговая рекомендация

| Тип признака | Что делать? | Почему? |
| --- | --- | --- |
| Бинарные (0/1) | Не стандартизировать | Они уже в подходящем виде. |
| Большие числовые дискретные значения | Стандартизировать | Улучшает обучение нейросетей и стабилизирует градиенты. |
| Категориальные (например, номер недели) | Кодировать (one-hot, embedding), масштаб не нужен | Чтобы не вводить ложные зависимости. |

---

▌Важные моменты

- Общая практика — масштабировать только числовые признаки, а бинарные и категориальные кодировать.
- Для нейросетей очень важно масштабировать числовые признаки, чтобы ускорить обучение и сделать его стабильнее.
- Для градиентных бустингов — обычно достаточно, чтобы числовые признаки были в разумных диапазонах, но не обязательно стандартизировать.

---

▌Итог

- Бинарные признаки — оставить как есть.
- Большие числовые признаки — стандартизировать (например, через StandardScaler).
- Категориальные — кодировать, масштабировать не нужно.